# Western Downs — Project Status Transition EDA

First empirical look at the framework, using the AEMO Generation Information files.

**What this notebook does:**
1. Load all AEMO Gen Info releases in `data/projects/aemo_geninfo/`
2. Tag each project to a transmission catchment via `data/rez/transmission_catchment_lookup.csv`
3. Detect status transitions between consecutive releases
4. Plot the Western Downs (WD) catchment timeline
5. Compare against the other four Southern QLD catchments (SD, DD, TG, WG)

**What this is not:**
- Not a Hawkes fit yet — that's for later, when we have more releases. With only 7 releases over 5 years, the temporal resolution is too coarse for a real intensity estimate.
- Not a causal claim — descriptive only.
- Not a complete event log — only the 7 releases we have. The narrative will improve substantially once the missing quarterly releases are sourced via Wayback Machine.

**Catchment tags reflect professional judgement on shared transmission topology, not formal Powerlink REZ membership.** The grid is meshed; REZ polygon boundaries don't determine constraint propagation. See `data/rez/transmission_catchment_lookup.csv` for the lookup.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from nem_herding.projects import (
    load_all_releases, join_catchment, detect_status_transitions,
    latest_project_snapshot,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)

# Find repo root by walking up until we see pyproject.toml
p = Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
REPO = p
DATA = REPO / 'data'
print(f'Repo root: {REPO}')

## 1. Load and inspect the panel

In [ ]:
panel = load_all_releases(DATA / 'projects' / 'aemo_geninfo')
panel = join_catchment(panel, DATA / 'rez' / 'transmission_catchment_lookup.csv')

print(f'Releases:           {sorted(panel["release_date"].unique())}')
print(f'Total project-rows: {len(panel):,}')
print(f'Unique site names:  {panel["site_name"].nunique():,}')

In [ ]:
# Catchment distribution at the latest release
latest = panel[panel['release_date'] == panel['release_date'].max()]
print(f'Latest snapshot: {latest["release_date"].iloc[0].date()}\n')
by_catch = (
    latest.groupby('transmission_catchment')
    .agg(n_projects=('site_name', 'nunique'),
         total_mw=('nameplate_mw', 'sum'))
    .sort_values('total_mw', ascending=False)
)
print(by_catch)

## 2. Detect status transitions across releases

A transition is a change in `status_bucket` for the same project between two consecutive releases.
These are the *events* the framework wants to track.

With 7 releases, the maximum possible transitions per project is 6. The annual cadence of the
available releases (mostly July of each year) means we'll under-count transitions that happened
and reverted within 12 months.

In [ ]:
transitions = detect_status_transitions(panel)
print(f'Total transitions detected: {len(transitions)}')
print('\nBy to_status:')
print(transitions['to_status'].value_counts())

## 3. Western Downs catchment — the focal case

In [ ]:
wd = transitions[transitions['transmission_catchment'] == 'WD'].copy()
wd = wd.sort_values('to_release')
print(f'Western Downs transitions: {len(wd)}\n')
print(wd[['site_name', 'from_release', 'to_release',
          'from_status', 'to_status', 'nameplate_mw']].to_string(index=False))

In [ ]:
# Cumulative committed capacity in Western Downs across releases
wd_panel = panel[panel['transmission_catchment'] == 'WD'].copy()
by_release = (
    wd_panel.groupby(['release_date', 'status_bucket'])['nameplate_mw']
    .sum().unstack(fill_value=0)
)
# Reorder columns so the stack reads left-to-right as the buildout pipeline
status_order = ['Proposed', 'Anticipated', 'Committed',
                'Existing less Announced Withdrawal',
                'Announced Withdrawal']
by_release = by_release[[c for c in status_order if c in by_release.columns]]
print(by_release.round(0))

In [ ]:
# Plot — capacity by status bucket across releases
fig, ax = plt.subplots(figsize=(11, 5.5))
by_release.plot(kind='area', stacked=True, ax=ax, alpha=0.78)
ax.set_title('Western Downs catchment — capacity by status across AEMO releases',
             fontsize=12, loc='left')
ax.set_xlabel('AEMO release date')
ax.set_ylabel('Capacity (MW)')
ax.legend(title='Status bucket', loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 3a. Phantom-project filtering

The AEMO Generation Information register is a survey-based publication: anything submitted is listed. This includes:
- Speculative entries from shell developers with no balance sheet
- 'Projects' that have sat in Proposed across 4+ releases with no progression  
- Offshore wind (no Australian project has reached FID at time of writing)
- Generic-named 'Energy Hub' / 'Renewable Energy Park' entries where owner ≡ site

These are flagged in the lookup with `phantom_risk` 0 (clean), 1 (elevated, review), 2 (high, exclude from headline).

The framework's primary analysis defaults to **phantom_risk < 2**. The 'all projects' view is kept as a secondary view — the difference between them is itself a research artifact, because phantom projects still influence AEMO's planning processes, transmission forecasts, and market participant expectations.

In [ ]:
# Reload the catchment lookup with phantom-risk columns
import pandas as pd
lookup = pd.read_csv(DATA / 'rez' / 'transmission_catchment_lookup.csv')
print(f'Lookup columns: {list(lookup.columns)}')
print(f'\nPhantom-risk distribution:')
print(lookup["phantom_risk"].value_counts().sort_index().rename({0: 'clean', 1: 'elevated', 2: 'high'}))
print(f'\nOwner-tier distribution:')
print(lookup['owner_tier'].value_counts())

In [ ]:
# Apply phantom filter to the panel
panel = panel.merge(
    lookup[['site_name','phantom_risk','owner','owner_tier']],
    on='site_name', how='left', suffixes=('', '_lk')
)
panel['phantom_risk'] = panel['phantom_risk'].fillna(0).astype(int)

print('Panel shape:', panel.shape)
print('\nPhantom risk in panel (all rows):')
print(panel['phantom_risk'].value_counts().sort_index())

In [ ]:
# Compare WD headline: all vs phantom-excluded
wd_all = panel[panel['transmission_catchment']=='WD']
wd_clean = panel[(panel['transmission_catchment']=='WD') & (panel['phantom_risk']<2)]

compare = pd.concat([
    wd_all.groupby('release_date')['nameplate_mw'].sum().rename('all_listed_mw'),
    wd_clean.groupby('release_date')['nameplate_mw'].sum().rename('phantom_excluded_mw'),
], axis=1)
compare['phantom_share'] = (compare['all_listed_mw'] - compare['phantom_excluded_mw']) / compare['all_listed_mw']
print('Western Downs catchment — capacity by release with/without phantoms:\n')
print(compare.round({'all_listed_mw': 0, 'phantom_excluded_mw': 0, 'phantom_share': 3}))

In [ ]:
# Plot: side-by-side stacked area for WD, with/without phantoms
import matplotlib.pyplot as plt

status_order = ['Proposed', 'Anticipated', 'Committed',
                'Existing less Announced Withdrawal', 'Announced Withdrawal']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (label, sub) in zip(axes, [('All listed projects', wd_all),
                                    ('Phantoms excluded (phantom_risk < 2)', wd_clean)]):
    series = (sub.groupby(['release_date','status_bucket'])['nameplate_mw']
              .sum().unstack(fill_value=0))
    cols = [c for c in status_order if c in series.columns]
    if cols:
        series[cols].plot(kind='area', stacked=True, ax=ax, alpha=0.78, legend=False)
    ax.set_title(f'Western Downs — {label}', fontsize=11, loc='left')
    ax.set_xlabel('AEMO release')
    ax.set_ylabel('Capacity (MW)' if ax==axes[0] else '')
    ax.grid(alpha=0.3)
axes[1].legend(title='Status', loc='upper left', fontsize=9)
fig.suptitle('Western Downs catchment: effect of phantom-project filter', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Latest snapshot comparison across all 5 Southern QLD catchments
latest_dt = panel['release_date'].max()
latest = panel[panel['release_date']==latest_dt]
sthn = latest[latest['transmission_catchment'].isin(['WD','SD','DD','TG','WG'])]

tbl = pd.DataFrame({
    'all_listed_mw': sthn.groupby('transmission_catchment')['nameplate_mw'].sum(),
    'phantom_excluded_mw': sthn[sthn['phantom_risk']<2]
        .groupby('transmission_catchment')['nameplate_mw'].sum(),
})
tbl['phantom_share'] = (tbl['all_listed_mw'] - tbl['phantom_excluded_mw']) / tbl['all_listed_mw']
tbl = tbl.fillna(0).round({'all_listed_mw': 0, 'phantom_excluded_mw': 0, 'phantom_share': 3})
print(f'Southern QLD as at {latest_dt.date()}:\n')
print(tbl)

## 4. Comparator view — all five Southern QLD catchments

If the synchronisation pattern only appears in Western Downs, that's a Western-Downs-specific story.
If it appears across all five Southern QLD catchments, the mechanism is more general — which is the
more interesting (and more defensible) finding.

In [ ]:
southern = panel[panel['transmission_catchment'].isin(['WD','SD','DD','TG','WG'])].copy()

fig, axes = plt.subplots(5, 1, figsize=(11, 14), sharex=True)
labels = {'WD': 'Western Downs', 'SD': 'Southern Downs',
          'DD': 'Darling Downs', 'TG': 'Tarong', 'WG': 'Woolooga'}
for ax, code in zip(axes, ['WD', 'SD', 'DD', 'TG', 'WG']):
    sub = southern[southern['transmission_catchment'] == code]
    series = (
        sub.groupby(['release_date', 'status_bucket'])['nameplate_mw']
        .sum().unstack(fill_value=0)
    )
    cols = [c for c in status_order if c in series.columns]
    if cols:
        series[cols].plot(kind='area', stacked=True, ax=ax, alpha=0.78, legend=False)
    ax.set_title(f'{labels[code]} ({code})', loc='left', fontsize=11)
    ax.set_ylabel('Capacity (MW)')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('AEMO release date')
axes[0].legend(title='Status', loc='upper left', fontsize=8)
fig.suptitle('Southern QLD transmission catchments — capacity by status', y=1.00)
fig.tight_layout()
plt.show()

## 5. Transition events — Southern QLD overview

The Hawkes process the framework will eventually fit treats each status transition as an event.
This is the raw event log.

In [ ]:
sthn_trans = transitions[transitions['transmission_catchment'].isin(['WD','SD','DD','TG','WG'])].copy()
sthn_trans = sthn_trans.sort_values(['to_release', 'transmission_catchment', 'site_name'])

print(f'Southern QLD transitions: {len(sthn_trans)}\n')
print(sthn_trans[['to_release','transmission_catchment','site_name',
                  'from_status','to_status','nameplate_mw']].to_string(index=False))

In [ ]:
# Transition counts by catchment x to_status
pivot = (
    sthn_trans.groupby(['transmission_catchment', 'to_status']).size()
    .unstack(fill_value=0)
)
print(pivot)

## 6. Sanity-check: what's still Unclassified?

These are QLD projects the keyword rules couldn't classify. They need manual review
in `data/rez/transmission_catchment_lookup.csv`. None of the headline results above
depend on these — they're held aside.

In [ ]:
unclassified = (
    panel[panel['transmission_catchment'] == 'Unclassified']
    .groupby('site_name')['nameplate_mw'].max()
    .sort_values(ascending=False)
)
print(f'Unclassified projects: {len(unclassified)}\n')
print(unclassified.head(30))

## Next steps

1. **Review `transmission_catchment_lookup.csv`** — particularly the ~80 Unclassified entries and the ~50 medium-confidence ones. Flip any you disagree with.
2. **Source more AEMO releases via Wayback Machine** — the missing quarters (April/Jan/Oct etc.) would lift transition resolution from annual to quarterly.
3. **Add a manual project-aliases file** — "Western Downs Green Power Hub" vs "Western Downs Green Power Hub P/L" are the same project; the panel currently treats them as separate. A small `project_aliases.csv` (site_name → canonical_project) would dedupe these.
4. **Connect to constraint binding data** — your existing `nem-constraints` work has the constraint-binding timestamps. Overlaying constraint binding intensity onto the buildout chart above is the bridge between the herding hypothesis and the operational impact.